# M6 · Retain information consistently

Persistence extends data lifetime, and a transaction makes related writes atomic. A supplied temporary database keeps deliberate failures away from regular application data.

**Start:** Use a Python 3 kernel; these cells use only the standard library. Restart the kernel and run top to bottom. All teaching Python is in the editable cells below; no repository imports or external lesson loaders are needed.

**Explore:** Predict → run → observe → change one input. Expected results assume the supplied values and a fresh run; changing earlier state may change later results. To return to the baseline, restart and rerun from the start.

**Temporary resources:** The setup creates a fresh disposable SQLite file. The final storage step removes that experiment. If you stop early, rerun the cleanup cell when finished. Restart and run all for a new baseline; do not repeatedly insert the same stored ID or recreate the same tables within one run.

## Exploration 1 — build and observe the mechanism

### 1. Choose an isolated persistent resource

A file can outlive a connection. This supplied setup creates a throwaway database for the exploration.

**Before running:** Predict the result or state change.

In [ ]:
import sqlite3
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

directory = TemporaryDirectory()
database_path = Path(directory.name) / "shop.sqlite"
print(database_path.name, database_path.exists())


**Change and explain:** Explain how a file path differs from a live database connection.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The path is chosen, but the database file is not created until a connection opens it.

</details>

### 2. Manage connections deliberately

Cleanup and transaction outcome have separate responsibilities. The helper closes connections and enables relationships on every open.

**Before running:** Predict the result or state change.

In [ ]:
def connect():
    connection = sqlite3.connect(database_path)
    connection.row_factory = sqlite3.Row
    connection.execute("PRAGMA foreign_keys = ON")
    return closing(connection)

with connect() as connection:
    connection.execute("CREATE TABLE products (product_id TEXT PRIMARY KEY, name TEXT, price_paise INTEGER)")
    print(connection.execute("SELECT COUNT(*) FROM products").fetchone()[0])
print(database_path.exists())


**Change and explain:** Locate the close responsibility; with connection later handles transactions, not closure.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The products table has zero rows; its database file exists after this connection closes.

</details>

### 3. Bind values and commit a complete insertion

Values are data, not SQL instructions. A successful transaction retains the written rows.

**Before running:** Predict the result or state change.

In [ ]:
with connect() as connection:
    with connection:
        connection.executemany("INSERT INTO products VALUES (?, ?, ?)", [
            ("1", "USB-C Cable", 49900),
            ("3", "Water Bottle", 89900),
        ])
    print(connection.execute("SELECT COUNT(*) FROM products").fetchone()[0])


**Change and explain:** Explain the different roles of the inner and outer context managers.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Two product rows remain after the transaction commits.

</details>

### 4. Translate a row and handle absence

Storage rows and application records have different interfaces. Check absence before reading fields.

**Before running:** Predict the result or state change.

In [ ]:
def get_product(product_id):
    with connect() as connection:
        row = connection.execute(
            "SELECT * FROM products WHERE product_id = ?", (product_id,),
        ).fetchone()
    if row is None:
        raise KeyError(product_id)
    return dict(row)

print(get_product("1"))


**Change and explain:** Catch the specific missing-product failure for ID 999.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Named fields become an ordinary dictionary after the connection has closed.

</details>

### 5. Connect headers and historical lines

Related tables connect identities; snapshots retain checkout-time facts. The schema is supplied so attention stays on the behavior.

**Before running:** Predict the result or state change.

In [ ]:
with connect() as connection:
    connection.executescript("""
        CREATE TABLE orders (order_id TEXT PRIMARY KEY, total_paise INTEGER);
        CREATE TABLE order_lines (
            order_id TEXT REFERENCES orders(order_id), product_id TEXT REFERENCES products(product_id),
            name TEXT, unit_price_paise INTEGER, quantity INTEGER CHECK (quantity > 0)
        );
    """)
    print([row[0] for row in connection.execute("SELECT name FROM sqlite_master WHERE type='table'")])


**Change and explain:** Which fields preserve history independently of current product records?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Products, orders, and order_lines are separate related tables.

</details>

### 6. Save related writes within one transaction

The header and all lines represent one operation. Let a failure escape the transaction so earlier writes roll back.

**Before running:** Predict the result or state change.

In [ ]:
def save_order(order_id, lines):
    total = sum(line[2] * line[3] for line in lines)
    with connect() as connection:
        with connection:
            connection.execute("INSERT INTO orders VALUES (?, ?)", (order_id, total))
            for product_id, name, price, quantity in lines:
                connection.execute("INSERT INTO order_lines VALUES (?, ?, ?, ?, ?)",
                                   (order_id, product_id, name, price, quantity))
    return total

print(save_order("demo-order", [("1", "USB-C Cable", 49900, 2)]))


**Change and explain:** Predict what should remain if the second line fails after the header and first line.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The complete cable order is committed with total 99800. Rerunning the same ID intentionally rejects a duplicate.

</details>

## Exploration 2 — change a condition and inspect the consequence

### 7. Reopen and retrieve the snapshot

A new connection reads retained facts rather than relying on the earlier in-memory variables.

**Before running:** Predict the result or state change.

In [ ]:
def load_lines(order_id):
    with connect() as connection:
        return [dict(row) for row in connection.execute(
            "SELECT * FROM order_lines WHERE order_id = ?", (order_id,),
        )]

saved_lines = load_lines("demo-order")
print(saved_lines[0]["unit_price_paise"], saved_lines[0]["quantity"])


**Change and explain:** Identify the separate connection opened for this retrieval.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Price 49900 and quantity 2 survive the earlier connection closing.

</details>

### 8. Update current data without rewriting history

The catalog’s current price and an order’s earlier price have different lifetimes.

**Before running:** Predict the result or state change.

In [ ]:
with connect() as connection:
    with connection:
        connection.execute("UPDATE products SET price_paise = ? WHERE product_id = ?", (99900, "1"))
print(get_product("1")["price_paise"])
print(load_lines("demo-order")[0]["unit_price_paise"])


**Change and explain:** Calculate the historical order total from its retained line values.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

Today’s price is 99900; the earlier line still uses 49900.

</details>

### 9. Supply a failure after an earlier write

A failure before any insertion cannot demonstrate undoing a partial save. This supplied hook rejects the bottle line in the temporary database.

**Before running:** Predict the result or state change.

In [ ]:
with connect() as connection:
    connection.executescript("""
        CREATE TRIGGER reject_bottle BEFORE INSERT ON order_lines
        WHEN NEW.product_id = '3'
        BEGIN SELECT RAISE(ABORT, 'simulated later-line failure'); END;
    """)
    print(connection.execute("SELECT name FROM sqlite_master WHERE type='trigger'").fetchone()[0])


**Change and explain:** Predict both row counts for the failed order before running the next two cells.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The temporary database now contains the supplied failure hook; the regular shop is untouched.

</details>

### 10. Observe the rejected multi-write save

Catching an exception communicates failure. The transaction is what restores consistent data.

**Before running:** Predict the result or state change.

In [ ]:
failed_lines = [("1", "USB-C Cable", 49900, 1), ("3", "Water Bottle", 89900, 1)]
try:
    save_order("failed-order", failed_lines)
except sqlite3.IntegrityError as error:
    print(type(error).__name__, str(error))


**Change and explain:** Explain where the exception leaves the transaction context.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The second line raises IntegrityError after earlier writes were attempted.

</details>

### 11. Inspect the consequence, not only the exception

Atomicity is a claim about resulting state. Inspect both parts of the failed operation.

**Before running:** Predict the result or state change.

In [ ]:
with connect() as connection:
    for table in ("orders", "order_lines"):
        # These table names are supplied constants; the external value is bound.
        count = connection.execute(
            f"SELECT COUNT(*) FROM {table} WHERE order_id = ?", ("failed-order",),
        ).fetchone()[0]
        print(table, count)
print(len(load_lines("demo-order")))


**Change and explain:** Why must the previously successful order remain?

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The failed order has zero headers and zero lines; the earlier order still has one line.

</details>

### 12. Close resources and remove only the experiment

Closing a connection is not deleting persistent history. Removing this deliberately temporary folder is separate cleanup.

**Before running:** Predict the result or state change.

In [ ]:
print("Before cleanup:", database_path.exists())
directory.cleanup()
print("After cleanup:", database_path.exists())


**Change and explain:** In Nano-Zon, choose a lasting database path instead of this temporary setup.

<details>
<summary>Reasoned expectation — reveal after predicting</summary>

The demonstration file is removed; no regular application database is modified.

</details>

## Apply the mechanism in Nano-Zon

Follow the [M6 lab guide](../../projects/nano-zon/labs/m06/README.md). The codealong isolates the mechanism; the lab combines it with the supplied application.

State the principle, point to the code that expresses it, then explain one limit of the observed result.

**Authoring status:** This notebook contains editable teaching source, not captured execution output. Syntax and structure can be inspected independently of execution; runtime delivery review remains pending.